<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Memory-Systeme
</b></font> </br></p>

---

In [2]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M16-Memory-Systeme"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, install_packages, mermaid, load_prompt
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.10
langchain-chroma                         1.1.0
langchain-classic                        1.0.1
langchain-community                      0.4.1
langchain-core                           1.2.16
langchain-ollama                         1.0.1
langchain-openai                         1.1.10
langchain-text-splitters                 1.1.1
langgraph                                1.0.9
langgraph-checkpoint                     4.0.0
langgraph-prebuilt                       1.0.8
langgraph-sdk                            0.3.9

IP-Adresse: 34.23.173.54
Hostname: 54.173.23.34.bc.googleusercontent.com
Stadt: North Charleston
Region: South Carolina
Land: US
Koordinaten: 32.8546,-79.9748
Provider: AS396982 Google LLC
Postleitzahl: 29415
Zeitzone: America/New_Yor

<p><font color='black' size="5">
Lernziele
</font></p>


Nach diesem Modul kannst du:

- **Kurzzeit-Memory** einsetzen: Conversation Buffer, Sliding Window, Summarization
- **Langzeit-Memory** aufbauen: Vektordatenbank und Entity Memory
- **Per-User-Memory** mit Thread-IDs realisieren
- Verschiedene Memory-Strategien kombinieren

---

## Warum brauchen Agenten Memory?

Ein LLM hat kein eingebautes Gedächtnis. Ohne explizite Mechanismen beginnt jedes Gespräch von vorne.

| Problem | Lösung |
|---------|--------|
| Gesprächsverlauf geht verloren | Kurzzeit-Memory im State |
| Kontext wächst unbegrenzt | Sliding Window oder Summarization |
| Nutzerdaten nach Session weg | Langzeit-Memory (Vektordatenbank) |
| Kein Personalisierungspotenzial | Entity Memory oder Per-User-Store |

In [3]:
import os
import operator
from typing import TypedDict, Annotated, Optional

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain.chat_models import init_chat_model
from langchain_core.messages import trim_messages, RemoveMessage, SystemMessage, HumanMessage, AIMessage

try:
    from genai_lib.utilities import mprint, mermaid, setup_api_keys
    setup_api_keys(["OPENAI_API_KEY"])
except ImportError:
    from IPython.display import display, Markdown
    def mprint(t): display(Markdown(t))
    def mermaid(d): print(d)

llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)
mprint("✅ Setup abgeschlossen")

✓ OPENAI_API_KEY erfolgreich gesetzt


✅ Setup abgeschlossen

In [4]:
mermaid('''
flowchart TB
    M[Memory-Systeme] --> K[Kurzzeit-Memory]
    M --> L[Langzeit-Memory]

    K --> K1["Conversation Buffer<br/>Voller Verlauf"]
    K --> K2["Sliding Window<br/>Letzte N Nachrichten"]
    K --> K3["Summarization<br/>Komprimierter Verlauf"]

    L --> L1["Semantisches Memory<br/>Vektordatenbank"]
    L --> L2["Entity Memory<br/>Key-Value Struktur"]
''')

---
## 1 Kurzzeit-Memory

Drei Strategien, die sich darin unterscheiden, wie viel Verlauf aufbewahrt wird.

### 1.1 Conversation Buffer – Vollständiger Verlauf

In [5]:
class BufferState(TypedDict):
    messages: Annotated[list, add_messages]

def buffer_chat(state: BufferState) -> BufferState:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

graph = StateGraph(BufferState)
graph.add_node("chat", buffer_chat)
graph.add_edge(START, "chat")
graph.add_edge("chat", END)

buffer_app = graph.compile(checkpointer=MemorySaver())
mprint("✅ Conversation Buffer aufgebaut")

✅ Conversation Buffer aufgebaut

In [6]:
config = {"configurable": {"thread_id": "buffer-demo"}}

for nachricht in ["Mein Name ist Emma.", "Ich arbeite als Datenwissenschaftlerin.", "Wie heiße ich?"]:
    result = buffer_app.invoke(
        {"messages": [HumanMessage(content=nachricht)]},
        config=config
    )
    antwort = result["messages"][-1].content
    mprint(f"**Nutzer:** {nachricht}  \n**Agent:** {antwort}")
    mprint("---")

**Nutzer:** Mein Name ist Emma.  
**Agent:** Hallo Emma! Wie kann ich dir helfen?

---

**Nutzer:** Ich arbeite als Datenwissenschaftlerin.  
**Agent:** Das klingt spannend, Emma! Datenwissenschaft ist ein faszinierendes Feld. An welchen Projekten arbeitest du gerade oder was interessiert dich besonders an deiner Arbeit?

---

**Nutzer:** Wie heiße ich?  
**Agent:** Du hast gesagt, dass dein Name Emma ist.

---

### 1.2 Sliding Window – Letzte N Nachrichten

`trim_messages()` begrenzt den Kontext auf die jüngsten Nachrichten, sobald ein Token-Limit erreicht wird.

In [7]:
class WindowState(TypedDict):
    messages: Annotated[list, add_messages]

def window_chat(state: WindowState) -> WindowState:
    trimmed = trim_messages(
        state["messages"],
        max_tokens=2000,
        strategy="last",
        token_counter=llm,
        include_system=True,
        allow_partial=False,
    )
    response = llm.invoke(trimmed)
    return {"messages": [response]}

graph2 = StateGraph(WindowState)
graph2.add_node("chat", window_chat)
graph2.add_edge(START, "chat")
graph2.add_edge("chat", END)
window_app = graph2.compile(checkpointer=MemorySaver())

# Veranschaulichung: trim_messages kuerzt
nachrichten = [HumanMessage(content=f"Nachricht {i}") for i in range(1, 8)]
gekuerzt = trim_messages(nachrichten, max_tokens=100, strategy="last", token_counter=llm)
mprint(f"Original: {len(nachrichten)} Nachrichten → Nach trim_messages: {len(gekuerzt)} Nachrichten")

Original: 7 Nachrichten → Nach trim_messages: 7 Nachrichten

### 1.3 Summarization Memory – Komprimierter Verlauf

Ältere Nachrichten werden vom LLM zusammengefasst statt verworfen. Der Kontext bleibt erhalten, der Token-Verbrauch wird begrenzt.

In [8]:
class SummaryState(TypedDict):
    messages: Annotated[list, add_messages]
    summary: str

def summarize_if_needed(state: SummaryState) -> SummaryState:
    # Komprimiert, sobald mehr als 6 Nachrichten vorhanden sind
    messages = state["messages"]
    if len(messages) < 6:
        return {}

    existing = state.get("summary", "")
    behalten = messages[-4:]
    zum_komprimieren = messages[:-4]

    prompt = (
        f"Fasse dieses Gespräch in 2-3 Sätzen zusammen.\n"
        f"Bisherige Zusammenfassung: {existing or 'Keine'}\n\n"
        + "\n".join(f"{m.type}: {m.content}" for m in zum_komprimieren)
    )
    neue_zusammenfassung = llm.invoke(prompt).content
    zu_entfernen = [RemoveMessage(id=m.id) for m in zum_komprimieren]
    summary_msg = SystemMessage(
        content=f"Bisheriger Verlauf (komprimiert): {neue_zusammenfassung}"
    )
    return {"messages": [summary_msg] + zu_entfernen, "summary": neue_zusammenfassung}

def summary_chat(state: SummaryState) -> SummaryState:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

graph3 = StateGraph(SummaryState)
graph3.add_node("summarize", summarize_if_needed)
graph3.add_node("chat", summary_chat)
graph3.add_edge(START, "summarize")
graph3.add_edge("summarize", "chat")
graph3.add_edge("chat", END)
summary_app = graph3.compile(checkpointer=MemorySaver())
mprint("✅ Summarization Memory aufgebaut")

✅ Summarization Memory aufgebaut

In [9]:
config = {"configurable": {"thread_id": "summary-demo"}}
themen = [
    "Mein Name ist Max, ich bin Softwareentwickler.",
    "Ich arbeite bei einer Berliner KI-Firma.",
    "Mein Lieblingswerkzeug ist Python.",
    "Ich interessiere mich für LangChain und LangGraph.",
    "Was weißt du über mich?",
]
for nachricht in themen:
    result = summary_app.invoke(
        {"messages": [HumanMessage(content=nachricht)]},
        config=config
    )
    antwort = result["messages"][-1].content
    n = len(result["messages"])
    mprint(f"**({n} Msgs) Nutzer:** {nachricht}  \n**Agent:** {antwort[:140]}...")
    mprint("---")

**(2 Msgs) Nutzer:** Mein Name ist Max, ich bin Softwareentwickler.  
**Agent:** Hallo Max! Schön, dich kennenzulernen. Was für Projekte oder Technologien interessieren dich als Softwareentwickler?...

---

**(4 Msgs) Nutzer:** Ich arbeite bei einer Berliner KI-Firma.  
**Agent:** Das klingt spannend! Berlin hat eine lebendige Tech- und KI-Szene. An welchen Projekten arbeitest du dort? Und welche Technologien oder Ansä...

---

**(6 Msgs) Nutzer:** Mein Lieblingswerkzeug ist Python.  
**Agent:** Python ist eine großartige Wahl, besonders für KI- und Machine Learning-Projekte! Es bietet eine Vielzahl von Bibliotheken wie TensorFlow, P...

---

**(6 Msgs) Nutzer:** Ich interessiere mich für LangChain und LangGraph.  
**Agent:** LangChain und LangGraph sind spannende Tools! LangChain ist besonders nützlich für die Entwicklung von Anwendungen, die auf Sprachmodellen b...

---

**(6 Msgs) Nutzer:** Was weißt du über mich?  
**Agent:** Ich weiß, dass du ein Softwareentwickler aus Berlin bist und dass du bei einer KI-Firma arbeitest. Python ist dein bevorzugtes Werkzeug, und...

---

---
## 2 Langzeit-Memory

Langzeit-Memory überlebt das Sitzungsende und steht in zukünftigen Gesprächen zur Verfügung.

### 2.1 Semantisches Memory – Vektordatenbank

In [10]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.tools import tool

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
memory_store = Chroma(
    collection_name="agent_memory",
    embedding_function=embeddings,
)

@tool
def memory_speichern(information: str) -> str:
    '''MEMORY SPEICHERN - Speichert eine wichtige Information dauerhaft.

    Verwende dieses Tool wenn der Nutzer Relevantes erwähnt:
    - Persönliche Fakten ("Ich arbeite als Arzt")
    - Präferenzen ("Ich mag kurze Antworten")
    - Ziele ("Ich lerne Python fuer Data Science")

    Args:
        information: Die Information in einem präzisen Satz

    Returns:
        Bestätigung der Speicherung
    '''
    memory_store.add_texts([information])
    return f"Gespeichert: {information}"

@tool
def memory_abrufen(frage: str) -> str:
    '''MEMORY ABRUFEN - Durchsucht das Gedächtnis nach relevanten Informationen.

    Args:
        frage: Suchanfrage in natürlicher Sprache

    Returns:
        Relevante gespeicherte Informationen
    '''
    docs = memory_store.similarity_search(frage, k=3)
    if not docs:
        return "Keine relevanten Informationen gefunden."
    return "\n".join(f"- {d.page_content}" for d in docs)

mprint("✅ Memory-Tools definiert")

✅ Memory-Tools definiert

In [11]:
from langchain.agents import create_agent

memory_agent = create_agent(
    model=llm,
    tools=[memory_speichern, memory_abrufen],
    system_prompt=(
        "Du bist ein hilfreicher Assistent mit Langzeitgedächtnis. "
        "Speichere wichtige Informationen über den Nutzer und rufe sie bei Bedarf ab."
    )
)

result1 = memory_agent.invoke({
    "messages": [HumanMessage(content="Ich heiße Lena und lerne KI-Programmierung.")]
})
mprint(f"**Agent:** {result1['messages'][-1].content}")

result2 = memory_agent.invoke({
    "messages": [HumanMessage(content="Was weißt du über mich?")]
})
mprint(f"**Agent:** {result2['messages'][-1].content}")

**Agent:** Ich habe die Informationen gespeichert: Du heißt Lena und lernst KI-Programmierung. Wie kann ich dir weiterhelfen?

**Agent:** Ich weiß, dass du Lena heißt und dass du KI-Programmierung lernst. Wenn es noch mehr gibt, das du teilen möchtest, lass es mich wissen!

### 2.2 Entity Memory – Key-Value für Entitäten

In [12]:
from pydantic import BaseModel, Field

class Entitaet(BaseModel):
    name: str = Field(description="Name der Entität (Person, Ort, Projekt)")
    beschreibung: str = Field(description="Kurze Beschreibung in einem Satz")

class EntitaetListe(BaseModel):
    entitaeten: list[Entitaet] = Field(default_factory=list)

class EntityState(TypedDict):
    messages: Annotated[list, add_messages]
    entity_memory: dict

extractor = llm.with_structured_output(EntitaetListe)

def entity_extraktor(state: EntityState) -> EntityState:
    # Extrahiert Entitaeten aus der letzten Nutzernachricht
    letzte = state["messages"][-1].content
    result = extractor.invoke(
        f"Extrahiere wichtige Entitäten (Personen, Projekte, Orte) aus:\n{letzte}"
    )
    updated = dict(state.get("entity_memory", {}))
    for e in result.entitaeten:
        updated[e.name] = e.beschreibung
    return {"entity_memory": updated}

def entity_chat(state: EntityState) -> EntityState:
    kontext = "\n".join(f"- {k}: {v}" for k, v in state.get("entity_memory", {}).items())
    system = f"Du kennst folgende Entitäten:\n{kontext}" if kontext else ""
    msgs = ([SystemMessage(content=system)] if system else []) + list(state["messages"])
    return {"messages": [llm.invoke(msgs)]}

graph4 = StateGraph(EntityState)
graph4.add_node("extraktor", entity_extraktor)
graph4.add_node("chat", entity_chat)
graph4.add_edge(START, "extraktor")
graph4.add_edge("extraktor", "chat")
graph4.add_edge("chat", END)
entity_app = graph4.compile(checkpointer=MemorySaver())
mprint("✅ Entity Memory aufgebaut")

✅ Entity Memory aufgebaut

In [13]:
config = {"configurable": {"thread_id": "entity-demo"}}
nachrichten = [
    "Ich arbeite mit Tom an Projekt Alpha.",
    "Tom ist unser Tech Lead und Projekt Alpha ist eine KI-Plattform.",
    "Was weißt du über Tom?",
]
for msg in nachrichten:
    result = entity_app.invoke(
        {"messages": [HumanMessage(content=msg)]},
        config=config
    )
    mem = result.get("entity_memory", {})
    antwort = result["messages"][-1].content
    mprint(f"**Nutzer:** {msg}")
    if mem:
        mprint(f"*Entity Memory: {mem}*")
    mprint(f"**Agent:** {antwort}")
    mprint("---")

**Nutzer:** Ich arbeite mit Tom an Projekt Alpha.

*Entity Memory: {'Tom': 'Eine Person, die an Projekt Alpha arbeitet.', 'Projekt Alpha': 'Ein Projekt, an dem Tom und ich arbeiten.'}*

**Agent:** Das klingt spannend! Wie läuft die Zusammenarbeit mit Tom an Projekt Alpha? Gibt es bestimmte Herausforderungen oder Erfolge, die du teilen möchtest?

---

**Nutzer:** Tom ist unser Tech Lead und Projekt Alpha ist eine KI-Plattform.

...

---

**Nutzer:** Was weißt du über Tom?

*Entity Memory: {'Tom': 'Eine Person, über die Informationen angefragt werden.', 'Projekt Alpha': 'Eine KI-Plattform.'}*

**Agent:** Ich habe keine spezifischen Informationen über Tom, da ich keine persönlichen Daten über Einzelpersonen habe. Wenn du mir mehr über seine Rolle, Erfahrungen oder Fähigkeiten erzählen möchtest, kann ich dir gerne helfen, basierend auf den Informationen, die du bereitstellst!

---

---
## 3 Per-User Memory

In Multi-User-Systemen trennt eine eindeutige `thread_id` pro Nutzer den Kontext vollständig.

In [14]:
def get_config(user_id: str, session_id: str = "default") -> dict:
    # Erstellt nutzerspezifische Konfiguration
    return {"configurable": {"thread_id": f"{user_id}:{session_id}"}}

config_alice = get_config("alice")
config_bob   = get_config("bob")

buffer_app.invoke(
    {"messages": [HumanMessage(content="Ich heiße Alice und mag Python.")]},
    config=config_alice
)
buffer_app.invoke(
    {"messages": [HumanMessage(content="Ich heiße Bob und arbeite mit Java.")]},
    config=config_bob
)

# Beide fragen nach ihrem Namen - vollstaendig getrennte Kontexte
for name, cfg in [("Alice", config_alice), ("Bob", config_bob)]:
    result = buffer_app.invoke(
        {"messages": [HumanMessage(content="Wie heiße ich?")]},
        config=cfg
    )
    mprint(f"**{name} fragt:** Wie heiße ich?  \n**Agent:** {result['messages'][-1].content}")

**Alice fragt:** Wie heiße ich?  
**Agent:** Du hast gesagt, dass du Alice heißt.

**Bob fragt:** Wie heiße ich?  
**Agent:** Du hast gesagt, dass du Bob heißt.

---
## Zusammenfassung

| Memory-Typ | Implementierung | Einsatz |
|-----------|----------------|---------|
| **Conversation Buffer** | `add_messages` Reducer | Kurze Gespräche, einfache Demos |
| **Sliding Window** | `trim_messages()` | Lange Gespräche, Token-Budget begrenzen |
| **Summarization** | `RemoveMessage` + LLM | Multi-Turn mit wichtigem Kontext |
| **Semantisches Memory** | ChromaDB + `@tool` | Nutzerpräferenzen über Sessions |
| **Entity Memory** | `with_structured_output` + Dict | CRM-Agenten, Support-Systeme |
| **Per-User Memory** | Thread-IDs im Checkpointer | Multi-User-Anwendungen |

**Faustregel:**
- **Kurzzeit immer** → Conversation Buffer oder Sliding Window
- **Langzeit bei Personalisierung** → Semantisches oder Entity Memory
- **Per-User** → eindeutige Thread-IDs + persistenter Checkpointer

Verwandte Module: M15 (Checkpointing), M18 (Human-in-the-Loop), M19 (Multi-Agent)